In [2]:
import pandas as pd

In [3]:
df = pd.read_excel('/content/Dataset for Data Analytics.xlsx')

In [4]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [5]:
df.shape

(1200, 14)

In [6]:
df.isnull().sum()

,0
OrderID,0
Date,0
CustomerID,0
Product,0
Quantity,0
UnitPrice,0
ShippingAddress,0
PaymentMethod,0
OrderStatus,0
TrackingNumber,0


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df['CouponCode'].dtypes

dtype('O')

In [10]:
df['CouponCode'].unique()

array(['SAVE10', 'FREESHIP', nan, 'WINTER15'], dtype=object)

In [11]:
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

In [12]:
df.describe()

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


In [13]:
numerical_summary = df[['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']].agg(['mean', 'median', 'count'])
print(numerical_summary)

           Quantity   UnitPrice  ItemsInCart  TotalPrice
mean       2.945833   356.41275        5.485   1053.9683
median     3.000000   364.21000        5.000    823.6150
count   1200.000000  1200.00000     1200.000   1200.0000


In [14]:
product_summary = df.groupby('Product').agg(
    Total_Revenue=('TotalPrice', 'sum'),
    Average_Spend=('TotalPrice', 'mean'),
    Total_Quantity=('Quantity', 'sum'),
    Order_Count=('OrderID', 'count')
).sort_values(by='Total_Revenue', ascending=False)
print(product_summary)

         Total_Revenue  Average_Spend  Total_Quantity  Order_Count
Product                                                           
Chair        195620.11    1098.989382             562          178
Printer      195612.61    1080.732652             542          181
Laptop       192126.56    1110.558150             535          173
Tablet       186568.95    1042.284637             497          179
Monitor      175651.41    1077.616012             480          163
Desk         167459.93     985.058412             508          170
Phone        151722.39     972.579423             411          156


In [15]:
referral_summary = df.groupby('ReferralSource')['TotalPrice'].sum().sort_values(ascending=False)
print(referral_summary)

ReferralSource
Instagram    275285.45
Email        261808.55
Google       250441.48
Facebook     250410.90
Referral     226815.58
Name: TotalPrice, dtype: float64


In [16]:
df['Date'] = pd.to_datetime(df['Date'])

In [17]:
df['YearMonth'] = df['Date'].dt.to_period('M')

In [18]:
monthly_trends = df.groupby('YearMonth').agg(
    Monthly_Revenue=('TotalPrice', 'sum'),
    Order_Volume=('OrderID', 'count')
).sort_index()


In [19]:
print(monthly_trends.head(10))

           Monthly_Revenue  Order_Volume
YearMonth                               
2023-01           56685.75            47
2023-02           40117.66            37
2023-03           48609.37            43
2023-04           27751.71            31
2023-05           63836.84            49
2023-06           49500.19            45
2023-07           42820.66            44
2023-08           54352.14            51
2023-09           29526.67            29
2023-10           52607.85            47


In [20]:
Q1 = df['TotalPrice'].quantile(0.25)
Q3 = df['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1

In [21]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [22]:
outliers = df[(df['TotalPrice'] < lower_bound) | (df['TotalPrice'] > upper_bound)]

In [23]:
print(f"Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")
print(f"Total number of outlier transactions found: {len(outliers)}")
print("\nTop 5 Highest Outlier Transactions:")
print(outliers[['OrderID', 'Product', 'Quantity', 'TotalPrice']].sort_values(by='TotalPrice', ascending=False).head())

Lower Bound: -1341.41, Upper Bound: 3330.41
Total number of outlier transactions found: 8

Top 5 Highest Outlier Transactions:
        OrderID  Product  Quantity  TotalPrice
789   ORD200789   Tablet         5     3456.40
1122  ORD201122  Monitor         5     3390.95
632   ORD200632   Laptop         5     3390.80
469   ORD200469    Chair         5     3384.90
328   ORD200328   Tablet         5     3370.20
